<a href="https://colab.research.google.com/github/ericj706/2026_Python/blob/main/day03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CNN실습시 필수: 코랩 오른쪽 상단에 -> 연결v 클릭하여 -> 런타임 유형 변경
    # T4 GPU( 그래픽 사용 ) 클릭 -> 저장 -> 연결

# 0) 실습에 필요한 데이터셋 (SVHN 칼라의 0부터 9까지의 이미지)
from torchvision import datasets
raw_train = datasets.SVHN( root='./data', split='train', download=True )
raw_test = datasets.SVHN( root='./data', split='test', download=True )

100%|██████████| 182M/182M [00:02<00:00, 71.2MB/s]
100%|██████████| 64.3M/64.3M [00:00<00:00, 101MB/s]


In [ ]:
# 0) GPU 연산 장치 설정
import torch # 'cuda' : 엔비디아 그래픽 드라이버 이름
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# 1) 파이토치의 텐서 타입으로 변경
# 0~255(밝기정도) 값들 0~1 정규화  # Y(타깃/정답지) 제외
X_train = torch.tensor(raw_train.data[:20000], dtype= torch.float32).to(device)/255.0   # 샘플로 2만개만 사용
Y_train = torch.tensor(raw_train.labels[:20000], dtype= torch.long).to(device)
X_test = torch.tensor(raw_test.data[:2000], dtype= torch.float32).to(device)/255.0    # 테스트 2000개만 사용
Y_test = torch.tensor(raw_test.labels[:2000], dtype= torch.long).to(device)
# 2) 신경망DNN 레이어 컨테이너 설정
import torch.nn as nn
model = nn.Sequential(
    # (1) 입력층 : 합성곱 -> ReLU -> 풀링
    # nn.Conv2d(채널수 , 특성개수 , 커널사이즈 , 패딩깊이)
    nn.Conv2d( in_channels=3, out_channels=16, kernel_size=3, padding=1 ),
    nn.ReLU(),
    # 만약에 앞전 합성곱 결과 특성맵 (16,32,32 ) 가정에 풀링 이후(stride가 2 이므로) ->  (16,16,16 )
    nn.MaxPool2d( kernel_size=2, stride=2 ),
    # (2) 은닉층 : 레이어2
    nn.Conv2d( in_channels=16, out_channels=32, kernel_size=3, padding=1 ),
    nn.ReLU(),
    nn.MaxPool2d( kernel_size=2, stride=2 ),
    # (3) 출력/분류기 레이어
    nn.Flatten() , # 32 * 8 * 8 (3차원) -> 2048(1차원) 으로 평탄화
    nn.Linear( in_features=32*8*8, out_features=64 ),
    nn.ReLU(),
    nn.Linear( in_features=64, out_features= 10)  # 최종 분류 레이어( 분류할 정답지/클래스 수 ) 0부터 9까지 10개
).to(device)
# 3) 손실함수: 다중분류이므로 소프트맥스 (확률 전체1 계산)
crierion = nn.CrossEntropyLoss()
# 4) 옵티마이저 ; optimizer = optim.AdamW( model.parameters(), lr=학습률)
import torch.optim as optim
optimizer = optim.AdamW( model.parameters(), lr=0.005)
# 5) 에포크(학습수) 단위 학습( 순전파 -> 오차계산 -> 역전파 -> 업데이트 )
for epoch in range(100):
    model.train()         # 1-학습모드
    optimizer.zero_grad()  # 2- 이전기울기 초기화
    logits = model(X_train) # 3- 순전파(학습)
    loss = crierion(logits, Y_train) # 4- 오차(학습과 정답 차이)
    loss.backward()  # 5- 역전파(오차 이용한 기울기/미분 가중치/편향 재설정 )
    optimizer.step()  # 6- 업데이트
    if epoch % 10 == 0 :
        print( epoch,": loss: ", loss.item() )
# 6) 모델 평가. model.train()학습모드 , model.eval() 평가/추론모드
model.eval()
with torch.no_grad():
    test_logits = model(X_test) # 테스트용 학습하기 (기울기 업데이트 안함)
    test_pred = torch.argmax(test_logits, dim=1)  # 확률중 가장 높은 값 추출 (소프트맥스)

from sklearn.metrics import accuracy_score
print(accuracy_score( Y_test.cpu().numpy(), test_pred.cpu().numpy() ))
# 7) 모델 추론 # [ [ 공부시간,결석,G1,G2 ] ]



cuda
0 : loss:  2.302685260772705
10 : loss:  2.2371842861175537
20 : loss:  2.2303307056427
30 : loss:  2.194347620010376
40 : loss:  2.0727269649505615
50 : loss:  1.8107576370239258
60 : loss:  1.4865779876708984
70 : loss:  1.243839144706726
80 : loss:  1.0610259771347046
90 : loss:  0.9249396324157715
0.7225
